# 02. 지연과 리뷰 — 핵심 질문> **배송이 늦으면 리뷰 점수가 실제로 떨어지는가. 며칠부터 급락하는가.**답이 나오면 그대로 운영 기준이 됩니다. "지연 N일을 넘기지 않는다"가 물류 SLA가 되니까요.

In [ ]:
import duckdb, pandas as pdimport matplotlib, matplotlib.pyplot as pltpd.set_option("display.max_columns", 50)# 한글 깨짐 방지for _f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR", "Noto Sans KR"]:    if _f in {f.name for f in matplotlib.font_manager.fontManager.ttflist}:        plt.rcParams["font.family"] = _f        breakplt.rcParams["axes.unicode_minus"] = False# DuckDB는 CSV를 그대로 읽습니다. 적재 과정이 따로 없습니다.con = duckdb.connect()con.execute(open("../sql/base.sql", encoding="utf-8").read().format(data="../data/raw"))def q(sql):    """SQL을 실행해 pandas DataFrame으로 돌려줍니다."""    return con.sql(sql).df()q("SELECT count(*) AS 주문수 FROM order_fact")

## 1. 먼저 분포를 본다구간을 나누기 전에 지연일이 실제로 어떻게 퍼져 있는지 봅니다.`sql/base.sql` 의 `delay_bucket` 구간은 지금 **임의로 정한 값** 이라, 이 분포를 보고 근거를 만들어 고쳐야 합니다.

In [ ]:
q("""SELECT count(*) AS 건수,       round(avg(delay_days), 1)  AS 평균,       median(delay_days)         AS 중앙값,       quantile_cont(delay_days, 0.25) AS p25,       quantile_cont(delay_days, 0.75) AS p75,       quantile_cont(delay_days, 0.95) AS p95,       max(delay_days)            AS 최대FROM order_factWHERE delay_days IS NOT NULL""")

In [ ]:
dist = q("""SELECT delay_days AS 지연일, count(*) AS 건수FROM order_factWHERE delay_days BETWEEN -40 AND 40GROUP BY 1 ORDER BY 1""")dist.plot(x="지연일", y="건수", kind="bar", figsize=(14, 4), legend=False, width=1.0)plt.title("배송 지연일 분포"); plt.xticks(rotation=0, fontsize=6); plt.tight_layout(); plt.show()

**판단할 것**: 위 분포를 보고 현재 구간(−8 / −1 / 0 / 7 / 20)이 타당한가요?바꿀 거라면 `sql/base.sql` 의 `delay_bucket` 을 고치고, 왜 그렇게 나눴는지 `docs/METRICS.md` 에 적으세요.

## 2. 구간별 리뷰 점수 — 이 프로젝트의 핵심 표

In [ ]:
delay = q("""SELECT delay_bucket AS 지연구간,       count(*) AS 주문수,       round(avg(review_score), 3) AS 평균점수,       round(100.0 * count(*) FILTER (WHERE review_score = 1) / count(*), 1) AS 최저점비율,       round(100.0 * count(*) FILTER (WHERE review_score >= 4) / count(*), 1) AS 만족비율FROM order_factWHERE delay_days IS NOT NULL AND has_reviewGROUP BY 지연구간ORDER BY 지연구간""")delay

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))delay.plot(x="지연구간", y="평균점수",  kind="bar", ax=ax[0], legend=False, color="#1F3864")ax[0].set_title("지연 구간별 평균 리뷰 점수"); ax[0].set_ylim(1, 5); ax[0].set_xlabel("")delay.plot(x="지연구간", y="최저점비율", kind="bar", ax=ax[1], legend=False, color="#C0392B")ax[1].set_title("지연 구간별 1점 리뷰 비율 (%)"); ax[1].set_xlabel("")for a in ax: a.tick_params(axis="x", rotation=45)plt.tight_layout(); plt.show()

**평균과 1점 비율을 따로 보는 이유**평균 점수는 완만하게 움직이는데 1점 비율은 급격히 튀는 구간이 있습니다.평균만 보면 그 지점을 놓칩니다. 운영에서 문제가 되는 건 평균이 아니라 **1점을 주고 떠나는 고객** 입니다.

## 3. 【직접 작성】 다른 요인을 고정해도 같은 패턴인가지금까지 본 건 **상관관계** 입니다.늦게 오는 주문은 부피가 크거나 원거리이거나 특정 셀러에 몰려 있을 수 있고,그 요인이 점수를 깎았을 수도 있습니다.**할 일**: 조건을 하나 고정하고도 같은 패턴이 남는지 확인하세요. 예를 들어- 카테고리 하나만 골라서 (`WHERE category = '...'`)- 같은 주(state) 안에서만 (`WHERE customer_state = 'SP'`)- 주문 금액대를 비슷하게 맞춰서패턴이 남으면 결론이 단단해지고, 사라지면 그 사실 자체가 중요한 발견입니다.

In [ ]:
# 여기에 작성하세요# 위 2번 쿼리에 WHERE 조건 하나를 더 붙여 비교해 보세요.

## 결론 작성아래 형식으로 한 문장을 만들어 `docs/FINDINGS.md` 에 옮기세요.**숫자 없는 문장은 쓰지 않습니다.**> 배송 지연 **[   ]** 일을 넘기는 구간부터 1점 리뷰 비율이 정시 대비 **[   ]** 배(**[  ]**% → **[  ]**%)로 증가한다.> 평균 점수는 **[   ]** 점에서 **[   ]** 점으로 하락한다.> 다만 이는 상관관계이며, **[ 통제한 요인 / 통제하지 못한 요인 ]** 이 남아 있다.